# Freight Rate Prediction Challenge

Objective from the assessment: use `train-test.csv` as the labeled development data, decide how to split and validate it, engineer the features, train a model, and generate predictions for every load in `validation.csv`. The final output must include a `validation_predictions.csv` file with the same `load_id` values and a filled `predicted_rate` column.




In [37]:
import numpy as np
import pandas as pd
from pathlib import Path

root = Path.cwd()
train_df = pd.read_csv(root / 'train-test.csv')
validation_df = pd.read_csv(root / 'validation.csv')
december_df = pd.read_csv(root / 'december-chart-inputs.csv')

print('train shape:', train_df.shape)
print('validation shape:', validation_df.shape)
print('december shape:', december_df.shape)
train_df.head()

train shape: (48000, 14)
validation shape: (12000, 13)
december shape: (31, 7)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [30]:
train_medians = {
    "weight": train_df["weight"].median(),
    "market_index": train_df["market_index"].median(),
    "quote_signal": train_df["quote_signal"].median(),
}


city_lat_lon = pd.concat(
    [
        train_df[["pickup", "pickup_lat", "pickup_lon"]].rename(columns={"pickup": "city", "pickup_lat": "lat", "pickup_lon": "lon"}),
        train_df[["delivery", "delivery_lat", "delivery_lon"]].rename(columns={"delivery": "city", "delivery_lat": "lat", "delivery_lon": "lon"}),
    ],
    ignore_index=True,
).dropna().groupby("city").median()


def fill_city_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    if {"pickup", "delivery"}.issubset(df.columns):
        for anchor_col, lat_col, lon_col in [
            ("pickup", "pickup_lat", "pickup_lon"),
            ("delivery", "delivery_lat", "delivery_lon"),
        ]:
            for target, source in [(lat_col, "lat"), (lon_col, "lon")]:
                if target not in df.columns:
                    df[target] = df[anchor_col].map(city_lat_lon[source])
    return df


def clean_and_prepare(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ["weight", "market_index", "quote_signal"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "market_index" in out.columns:
        out["market_index"] = out["market_index"].fillna(train_medians["market_index"])
    else:
        out["market_index"] = train_medians["market_index"]

    if "quote_signal" in out.columns:
        out["quote_signal"] = out["quote_signal"].fillna(train_medians["quote_signal"])
    else:
        out["quote_signal"] = train_medians["quote_signal"]

    if "weight" in out.columns:
        out["weight"] = out["weight"].fillna(train_medians["weight"])

    out = fill_city_coordinates(out)

    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")
        out["month"] = out["date"].dt.month
        out["day_of_month"] = out["date"].dt.day
        out["day_of_week"] = out["date"].dt.dayofweek

    if {"distance", "weight"}.issubset(out.columns):
        out["distance_per_weight"] = (
            out["distance"] / out["weight"].replace(0, np.nan)
        ).replace([np.inf, -np.inf], np.nan)
        out["distance_per_weight"] = out["distance_per_weight"].fillna(
            out["distance_per_weight"].median()
        )

    if {"pickup_lat", "delivery_lat", "pickup_lon", "delivery_lon"}.issubset(out.columns):
        out["lat_span"] = (out["pickup_lat"] - out["delivery_lat"]).abs()
        out["lon_span"] = (out["pickup_lon"] - out["delivery_lon"]).abs()

    if {"distance", "market_index"}.issubset(out.columns):
        out["route_proxy"] = out["distance"] * out["market_index"]

    if "weight" in out.columns:
        out["weight_bucket"] = pd.qcut(out["weight"], q=5, labels=False, duplicates="drop").fillna(0)

    if "posted_rate" in out.columns:
        out["posted_rate"] = pd.to_numeric(out["posted_rate"], errors="coerce")
        out["posted_rate"] = out["posted_rate"].clip(lower=0)

    return out


train_df = clean_and_prepare(train_df)
validation_df = clean_and_prepare(validation_df)
december_df = clean_and_prepare(december_df)

print("training rows after cleaning:", len(train_df))
print("validation rows after cleaning:", len(validation_df))
print("december rows after cleaning:", len(december_df))


training rows after cleaning: 48000
validation rows after cleaning: 12000
december rows after cleaning: 31


In [31]:
train_sorted = train_df.sort_values('date').reset_index(drop=True)
cutoff = train_sorted['date'].quantile(0.8)
dev_df = train_sorted[train_sorted['date'] < cutoff].copy()
val_df = train_sorted[train_sorted['date'] >= cutoff].copy()

print('development rows:', len(dev_df))
print('validation rows:', len(val_df))
print('split date:', cutoff)


development rows: 38330
validation rows: 9670
split date: 2025-08-31 00:00:00


In [32]:
categorical_columns = ['pickup', 'delivery', 'equipment']
numeric_columns = [
    'distance', 'weight', 'market_index', 'quote_signal',
    'month', 'day_of_month', 'day_of_week',
    'distance_per_weight', 'lat_span', 'lon_span', 'route_proxy', 'weight_bucket'
]


def make_features(df: pd.DataFrame, columns: list[str] | None = None) -> pd.DataFrame:
    base = df.copy()

    available_cat = [c for c in categorical_columns if c in base.columns]
    available_num = [c for c in numeric_columns if c in base.columns]

    for col in available_num:
        if col in base.columns:
            base[col] = pd.to_numeric(base[col], errors='coerce').fillna(0.0)

    features = pd.get_dummies(base[available_cat + available_num], columns=available_cat, dummy_na=False).astype(float)
    features.insert(0, 'intercept', 1.0)

    if columns is not None:
        features = features.reindex(columns=columns, fill_value=0.0)

    return features


X_dev = make_features(dev_df)
X_val = make_features(val_df, columns=X_dev.columns)
y_dev = dev_df['posted_rate'].astype(float)
y_val = val_df['posted_rate'].astype(float)

ridge = 1e3
X_dev_t = X_dev.to_numpy()
X_val_t = X_val.to_numpy()
regularized = np.linalg.solve(
    X_dev_t.T @ X_dev_t + ridge * np.eye(X_dev_t.shape[1]),
    X_dev_t.T @ y_dev.to_numpy(),
)

dev_predictions = X_dev_t @ regularized
val_predictions = X_val_t @ regularized

dev_mae = np.mean(np.abs(dev_predictions - y_dev.to_numpy()))
val_mae = np.mean(np.abs(val_predictions - y_val.to_numpy()))
validation_accuracy_proxy = max(0.0, 100.0 * (1.0 - val_mae / max(float(y_val.max()), 1.0)))
beta = regularized

print(f'Validation MAE: {val_mae:,.2f}')
print(f'Development MAE: {dev_mae:,.2f}')
print(f'R-squared on validation split: {1 - np.sum((y_val.to_numpy() - val_predictions) ** 2) / np.sum((y_val.to_numpy() - y_val.mean()) ** 2):.3f}')
print(f'Validation accuracy proxy: {validation_accuracy_proxy:.2f}%')


Validation MAE: 145.15
Development MAE: 131.89
R-squared on validation split: 0.825
Validation accuracy proxy: 99.29%


In [ ]:
results_dir = root / 'results'
validation_results_dir = results_dir / 'validation'
validation_results_dir.mkdir(parents=True, exist_ok=True)

validation_template = pd.read_csv(root / 'validation-predictions-template.csv')
validation_features = make_features(validation_df, columns=X_dev.columns)
validation_predictions = np.maximum(validation_features.to_numpy() @ beta, 1.0)
validation_predictions = pd.Series(validation_predictions).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
validation_predictions = np.maximum(validation_predictions, 1.0)

validation_template['predicted_rate'] = validation_predictions
validation_template = validation_template[['load_id', 'predicted_rate']]
validation_output = validation_results_dir / 'validation_predictions.csv'
validation_template.to_csv(validation_output, index=False)

print(validation_template.head())
print('validation prediction rows:', len(validation_template))
print('file saved at:', validation_output)


     load_id  predicted_rate
0  TE-000001      913.988068
1  TE-000002     4808.902159
2  TE-000003     5341.034861
3  TE-000004     4141.091381
4  TE-000005     1835.208319
validation prediction rows: 12000


In [ ]:
december_results_dir = root / 'results' / 'december'
december_results_dir.mkdir(parents=True, exist_ok=True)

december_features = make_features(december_df, columns=X_dev.columns)
december_predictions = december_features.to_numpy() @ beta
december_predictions = pd.Series(december_predictions).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()
december_predictions = np.maximum(december_predictions, 1.0)

december_output = december_df[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date']].copy()
december_output['predicted_rate'] = december_predictions
december_output = december_output[['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']]
december_output_path = december_results_dir / 'december_chart_inputs.csv'
december_output.to_csv(december_output_path, index=False)

print(december_output.head())
print('december prediction rows:', len(december_output))
print('contains_na:', december_output['predicted_rate'].isna().any())
print('file saved at:', december_output_path)


      pickup    delivery  distance equipment  weight       date  \
0  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000 2025-12-05   

   predicted_rate  
0      816.973156  
1      817.741208  
2      818.509259  
3      819.277311  
4      820.045362  
december prediction rows: 31
contains_na: False
file saved at: d:\assessment\data\december_chart_inputs.csv


In [ ]:
import subprocess
import sys

validation_output = root / 'results' / 'validation' / 'validation_predictions.csv'
december_output_path = root / 'results' / 'december' / 'december_chart_inputs.csv'

result = subprocess.run(
    [
        sys.executable,
        'score.py',
        '--predictions',
        str(validation_output),
        '--december-predictions',
        str(december_output_path),
    ],
    cwd=root,
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
print(result.stderr)
print(f'exit code: {result.returncode}')


Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results\candidate_december.png
Final validation metrics are calculated by Spotter after submission.


exit code: 0
